In [ ]:
import os
import numpy as np
import pandas as pd
import tensorflow as tf
import xgboost as xgb
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, RobustScaler
from sklearn.impute import SimpleImputer
from sklearn.metrics import roc_auc_score, precision_recall_curve, confusion_matrix, classification_report
import matplotlib.pyplot as plt
import seaborn as sns
import logging

# Configure Logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)

C:\Users\yashs\AppData\Local\Temp\ipykernel_16544\4083498419.py:2: DtypeWarning: Columns (2,3,6,9,26,27,28,29,30,36,41,105,131,132,134,135,136,137,141,152,159,163,169,173,242,243,244) have mixed types. Specify dtype option on import or set low_memory=False.
  all_df = pd.read_csv("../data/all_df.csv")


(51264, 268)

In [ ]:
class Config:
    """
    Configuration class to hold all hyperparameters and file paths.
    This ensures easy reproducibility and centralized control.
    """
    # Paths
    DATA_DIR = "../data"
    KOI_RESULTS_DIR = "results_koi"
    TOI_RESULTS_DIR = "results_toi"
    CSV_PATH = os.path.join(DATA_DIR, "all_df.csv")
    
    # Model Hyperparameters
    BATCH_SIZE = 32
    EPOCHS = 20
    LEARNING_RATE = 1e-4
    
    # Input Shapes for CNN
    # Global view: 2001 points (full orbit)
    # Local view: 201 points (transit zoom)
    GLOBAL_VIEW_SHAPE = (2001, 1)
    LOCAL_VIEW_SHAPE = (201, 1)
    
    # Random Seed for Reproducibility
    SEED = 42

In [ ]:
class ExoplanetDataGenerator(tf.keras.utils.Sequence):
    """
    Custom Data Generator to load light curve data from disk on-the-fly.
    This is critical for memory efficiency when handling thousands of .npz files.
    """
    def __init__(self, file_paths, labels, batch_size=32, shuffle=True):
        self.file_paths = file_paths
        self.labels = labels
        self.batch_size = batch_size
        self.shuffle = shuffle
        self.indices = np.arange(len(self.file_paths))
        self.on_epoch_end()

    def __len__(self):
        """Denotes the number of batches per epoch"""
        return int(np.floor(len(self.file_paths) / self.batch_size))

    def __getitem__(self, index):
        """Generate one batch of data"""
        # Generate indexes of the batch
        indexes = self.indices[index*self.batch_size:(index+1)*self.batch_size]
        
        # Find list of IDs
        batch_paths = [self.file_paths[k] for k in indexes]
        batch_labels = [self.labels[k] for k in indexes]
        
        # Generate data
        X, y = self.__data_generation(batch_paths, batch_labels)
        return X, y

    def on_epoch_end(self):
        """Updates indexes after each epoch"""
        if self.shuffle:
            np.random.shuffle(self.indices)

    def __data_generation(self, batch_paths, batch_labels):
        """Generates data containing batch_size samples"""
        # Initialization
        X_global = np.empty((self.batch_size, *Config.GLOBAL_VIEW_SHAPE))
        X_local = np.empty((self.batch_size, *Config.LOCAL_VIEW_SHAPE))
        y = np.empty((self.batch_size), dtype=int)

        for i, path in enumerate(batch_paths):
            try:
                with np.load(path) as data:
                    # Load views
                    xg = data['X_global']
                    xl = data['X_local']
                    
                    # Ensure correct dimensions (add channel dim if needed)
                    if xg.ndim == 1: xg = np.expand_dims(xg, axis=-1)
                    if xl.ndim == 1: xl = np.expand_dims(xl, axis=-1)
                    
                    # Handle potential shape mismatches by padding/truncating if necessary
                    # (Assuming preprocessing ensured correct shapes, but adding safety)
                    X_global[i,] = xg[:Config.GLOBAL_VIEW_SHAPE[0]]
                    X_local[i,] = xl[:Config.LOCAL_VIEW_SHAPE[0]]
                    y[i] = batch_labels[i]
            except Exception as e:
                logger.error(f"Error loading {path}: {e}")
                # Fallback to zeros to avoid crashing training
                X_global[i,] = np.zeros(Config.GLOBAL_VIEW_SHAPE)
                X_local[i,] = np.zeros(Config.LOCAL_VIEW_SHAPE)
                y[i] = 0 

        return [X_global, X_local], y

In [ ]:
class DataManager:
    """
    Handles data loading, cleaning, and splitting for both Tabular and Image data.
    """
    def __init__(self, config):
        self.config = config
        self.tabular_scaler = RobustScaler()
        self.imputer = SimpleImputer(strategy='median')
        
    def load_and_split_data(self):
        logger.info("Loading Master Catalog...")
        # low_memory=False prevents mixed type warnings for large CSVs
        df = pd.read_csv(self.config.CSV_PATH, low_memory=False)
        
        # Filter for rows that have corresponding downloaded .npz files
        valid_rows = []
        
        logger.info("Validating file existence...")
        for idx, row in df.iterrows():
            # Construct file path based on catalog
            if row['catalog'] == 'KOI':
                fname = f"KIC_{int(row['target_id'])}.npz"
                fpath = os.path.join(self.config.KOI_RESULTS_DIR, fname)
            else:
                fname = f"TIC_{int(row['target_id'])}.npz"
                fpath = os.path.join(self.config.TOI_RESULTS_DIR, fname)
            
            if os.path.exists(fpath):
                row['file_path'] = fpath
                valid_rows.append(row)
        
        if not valid_rows:
            logger.warning("No valid data files found. Ensure download_data.py has run.")
            return None, None
            
        df_valid = pd.DataFrame(valid_rows)
        logger.info(f"Found {len(df_valid)} valid samples.")
        
        # Extract Tabular Features
        X_tabular = self._extract_tabular_features(df_valid)
        y = df_valid['label'].values.astype(int)
        file_paths = df_valid['file_path'].values
        
        # Stratified Split
        X_train_paths, X_test_paths, X_train_tab, X_test_tab, y_train, y_test = train_test_split(
            file_paths, X_tabular, y, test_size=0.2, random_state=self.config.SEED, stratify=y
        )
        
        # Preprocess Tabular Data (Impute -> Scale)
        X_train_tab = self.imputer.fit_transform(X_train_tab)
        X_test_tab = self.imputer.transform(X_test_tab)
        
        X_train_tab = self.tabular_scaler.fit_transform(X_train_tab)
        X_test_tab = self.tabular_scaler.transform(X_test_tab)
        
        return (X_train_paths, X_train_tab, y_train), (X_test_paths, X_test_tab, y_test)

    def _extract_tabular_features(self, df):
        """
        Extracts and coalesces relevant stellar parameters from KOI/TOI columns.
        """
        data = pd.DataFrame(index=df.index)
        
        # Coalesce columns (KOI vs TOI naming conventions)
        # Period
        data['period'] = df['period'].fillna(df.get('koi_period', np.nan))
        # Duration
        data['duration'] = df.get('koi_duration', np.nan).fillna(df.get('duration', np.nan))
        # Planet Radius
        data['prad'] = df.get('koi_prad', np.nan).fillna(df.get('prad', np.nan))
        # Equilibrium Temp
        data['teq'] = df.get('koi_teq', np.nan).fillna(df.get('teq', np.nan))
        # Insolation Flux
        data['insol'] = df.get('koi_insol', np.nan).fillna(df.get('insol', np.nan))
        # Stellar Effective Temp
        data['steff'] = df.get('koi_steff', np.nan).fillna(df.get('st_teff', np.nan))
        # Stellar Surface Gravity
        data['slogg'] = df.get('koi_slogg', np.nan).fillna(df.get('st_logg', np.nan))
        # Stellar Radius
        data['srad'] = df.get('koi_srad', np.nan).fillna(df.get('st_rad', np.nan))
        
        return data.values

In [ ]:
class DualViewCNN:
    """
    A Dual-View CNN inspired by AstroNet.
    Processes Global View (full orbit) and Local View (transit) separately,
    then fuses them for classification.
    """
    @staticmethod
    def build(global_shape, local_shape):
        # --- Global View Branch ---
        input_global = tf.keras.layers.Input(shape=global_shape, name='global_input')
        x_g = tf.keras.layers.Conv1D(16, 5, activation='relu', padding='same')(input_global)
        x_g = tf.keras.layers.Conv1D(16, 5, activation='relu', padding='same')(x_g)
        x_g = tf.keras.layers.MaxPooling1D(5, strides=2)(x_g)
        x_g = tf.keras.layers.Conv1D(32, 5, activation='relu', padding='same')(x_g)
        x_g = tf.keras.layers.Conv1D(32, 5, activation='relu', padding='same')(x_g)
        x_g = tf.keras.layers.MaxPooling1D(5, strides=2)(x_g)
        x_g = tf.keras.layers.Flatten()(x_g)
        
        # --- Local View Branch ---
        input_local = tf.keras.layers.Input(shape=local_shape, name='local_input')
        x_l = tf.keras.layers.Conv1D(16, 5, activation='relu', padding='same')(input_local)
        x_l = tf.keras.layers.Conv1D(16, 5, activation='relu', padding='same')(x_l)
        x_l = tf.keras.layers.MaxPooling1D(5, strides=2)(x_l)
        x_l = tf.keras.layers.Conv1D(32, 5, activation='relu', padding='same')(x_l)
        x_l = tf.keras.layers.Conv1D(32, 5, activation='relu', padding='same')(x_l)
        x_l = tf.keras.layers.MaxPooling1D(5, strides=2)(x_l)
        x_l = tf.keras.layers.Flatten()(x_l)
        
        # --- Fusion ---
        x = tf.keras.layers.Concatenate()([x_g, x_l])
        x = tf.keras.layers.Dense(512, activation='relu')(x)
        x = tf.keras.layers.Dropout(0.5)(x) # Regularization
        x = tf.keras.layers.Dense(64, activation='relu')(x)
        output = tf.keras.layers.Dense(1, activation='sigmoid', name='cnn_output')(x)
        
        model = tf.keras.Model(inputs=[input_global, input_local], outputs=output, name="AstroNet_DualView")
        
        model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=Config.LEARNING_RATE),
                      loss='binary_crossentropy',
                      metrics=['accuracy', tf.keras.metrics.AUC(name='auc')])
        return model

In [ ]:
class StellarXGB:
    """
    Wrapper for XGBoost Classifier to handle tabular stellar metadata.
    """
    def __init__(self):
        self.model = xgb.XGBClassifier(
            n_estimators=200,
            learning_rate=0.05,
            max_depth=6,
            subsample=0.8,
            colsample_bytree=0.8,
            objective='binary:logistic',
            eval_metric='logloss',
            use_label_encoder=False,
            random_state=Config.SEED
        )
        
    def train(self, X_train, y_train, X_val, y_val):
        self.model.fit(
            X_train, y_train,
            eval_set=[(X_val, y_val)],
            early_stopping_rounds=20,
            verbose=False
        )
        
    def predict_proba(self, X):
        return self.model.predict_proba(X)[:, 1]

In [ ]:
class HybridExoplanetLearner:
    """
    Orchestrator class that manages the entire pipeline:
    1. Data Loading
    2. CNN Training
    3. XGBoost Training
    4. Ensemble Prediction & Evaluation
    """
    def __init__(self):
        self.config = Config()
        self.data_manager = DataManager(self.config)
        self.cnn_model = DualViewCNN.build(self.config.GLOBAL_VIEW_SHAPE, self.config.LOCAL_VIEW_SHAPE)
        self.xgb_model = StellarXGB()
        
    def run(self):
        # 1. Load Data
        train_data, test_data = self.data_manager.load_and_split_data()
        if train_data is None: return
        
        (X_train_paths, X_train_tab, y_train) = train_data
        (X_test_paths, X_test_tab, y_test) = test_data
        
        # 2. Train CNN
        logger.info("Training CNN Branch...")
        train_gen = ExoplanetDataGenerator(X_train_paths, y_train, batch_size=self.config.BATCH_SIZE)
        test_gen = ExoplanetDataGenerator(X_test_paths, y_test, batch_size=self.config.BATCH_SIZE, shuffle=False)
        
        callbacks = [
            tf.keras.callbacks.EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True),
            tf.keras.callbacks.ModelCheckpoint("best_cnn_model.h5", save_best_only=True)
        ]
        
        self.cnn_model.fit(
            train_gen,
            validation_data=test_gen,
            epochs=self.config.EPOCHS,
            callbacks=callbacks
        )
        
        # 3. Train XGBoost
        logger.info("Training XGBoost Branch...")
        # Create a validation split for XGBoost early stopping
        X_train_sub, X_val_sub, y_train_sub, y_val_sub = train_test_split(
            X_train_tab, y_train, test_size=0.1, random_state=self.config.SEED
        )
        self.xgb_model.train(X_train_sub, y_train_sub, X_val_sub, y_val_sub)
        
        # 4. Evaluation & Ensemble
        logger.info("Evaluating Hybrid Ensemble...")
        
        # Get Predictions
        # Note: We truncate test set to match generator's full batches to avoid shape mismatch
        n_samples = len(test_gen) * self.config.BATCH_SIZE
        
        cnn_preds = self.cnn_model.predict(test_gen).flatten()
        
        # Align tabular data and labels with the truncated generator output
        y_test_truncated = y_test[:n_samples]
        X_test_tab_truncated = X_test_tab[:n_samples]
        
        xgb_preds = self.xgb_model.predict_proba(X_test_tab_truncated)
        
        # --- Ensemble Logic (Weighted Average) ---
        # We can adjust weights based on validation performance
        ensemble_preds = (0.6 * cnn_preds[:n_samples]) + (0.4 * xgb_preds)
        
        # Metrics
        self._print_metrics(y_test_truncated, ensemble_preds, "Hybrid Ensemble")
        
    def _print_metrics(self, y_true, y_pred_proba, name):
        y_pred_class = (y_pred_proba > 0.5).astype(int)
        print(f"\n--- {name} Performance ---")
        print(classification_report(y_true, y_pred_class))
        print(f"ROC-AUC: {roc_auc_score(y_true, y_pred_proba):.4f}")
        
        # Plot Confusion Matrix
        cm = confusion_matrix(y_true, y_pred_class)
        plt.figure(figsize=(6,5))
        sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
        plt.title(f'{name} Confusion Matrix')
        plt.ylabel('True Label')
        plt.xlabel('Predicted Label')
        plt.show()

In [ ]:
if __name__ == "__main__":
    # Check if data exists before running
    if os.path.exists(Config.KOI_RESULTS_DIR) or os.path.exists(Config.TOI_RESULTS_DIR):
        print("Initializing Hybrid Learner...")
        learner = HybridExoplanetLearner()
        # learner.run() # Uncomment to start training once data is ready
        print("Model architecture loaded. Ready to run learner.run() once data download completes.")
    else:
        print("Data directories not found. Please wait for download_data.py to finish.")